<a href="https://colab.research.google.com/github/usmanumer038/ml-internship-work/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector & Leakage / Privacy Audit

**Lane:** Refresh / Content Opportunity Scoring  
**Feature window:** March 2026  
**Purpose:** Build the approved five-feature March feature matrix and audit it for future/target leakage, identifier leakage, excluded metadata, missing values, and invalid numeric values.

This notebook uses the same FlyRank Hugging Face + DuckDB access pattern as the working W07 notebook.

In [1]:
!pip -q install duckdb scikit-learn pandas matplotlib

import os
import json
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN not found. Add your read token to Google Colab Secrets "
        "with the name HF_TOKEN."
    )

con = duckdb.connect()

con.execute(
    f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

BASE = "hf://datasets/FlyRank/internship-warehouse"
TABLE = f"{BASE}/fact_content_daily_performance/**/*.parquet"

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

print("Connected to FlyRank Internship Warehouse")
print("Table:", TABLE)

Connected to FlyRank Internship Warehouse
Table: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet


## 1. Approved model features

The model uses only March aggregate behavioral/performance features. Identifiers and future outcome information are kept out of the model feature matrix.

In [2]:
FEATURES = [
    "impressions",
    "clicks",
    "sessions",
    "engaged_sessions",
    "engagement_rate"
]

print("Approved model features:")
for feature in FEATURES:
    print("-", feature)

Approved model features:
- impressions
- clicks
- sessions
- engaged_sessions
- engagement_rate


In [3]:
march = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        COALESCE(SUM(gsc_impressions), 0) AS impressions,
        COALESCE(SUM(gsc_clicks), 0) AS clicks,
        COALESCE(SUM(ga4_sessions), 0) AS sessions,
        COALESCE(SUM(ga4_engaged_sessions), 0) AS engaged_sessions
    FROM read_parquet('{TABLE}')
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print("March rows:", len(march))
print("Columns:")
print(list(march.columns))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March rows: 331437
Columns:
['client_hash_id', 'content_hash_id', 'impressions', 'clicks', 'sessions', 'engaged_sessions']


In [4]:
for column in [
    "impressions",
    "clicks",
    "sessions",
    "engaged_sessions"
]:
    march[column] = pd.to_numeric(
        march[column],
        errors="coerce"
    )

march["engagement_rate"] = np.where(
    march["sessions"] > 0,
    march["engaged_sessions"] / march["sessions"],
    0.0
)

march[FEATURES] = march[FEATURES].replace(
    [np.inf, -np.inf],
    np.nan
)

print("Engagement rate calculated.")

Engagement rate calculated.


## 2. Missing-value audit

Missing values in the source are not automatically leakage. They are audited first, then handled explicitly in the final feature matrix.

In [5]:
missing_before = march[FEATURES].isna().sum()

print("Missing values BEFORE cleaning:")
print(missing_before)

print(
    "\nTotal missing values BEFORE cleaning:",
    int(missing_before.sum())
)

march[FEATURES] = march[FEATURES].fillna(0)

missing_after = march[FEATURES].isna().sum()

print("\nMissing values AFTER cleaning:")
print(missing_after)

print(
    "\nTotal missing values AFTER cleaning:",
    int(missing_after.sum())
)

Missing values BEFORE cleaning:
impressions         0
clicks              0
sessions            0
engaged_sessions    0
engagement_rate     0
dtype: int64

Total missing values BEFORE cleaning: 0

Missing values AFTER cleaning:
impressions         0
clicks              0
sessions            0
engaged_sessions    0
engagement_rate     0
dtype: int64

Total missing values AFTER cleaning: 0


In [6]:
summary = march[FEATURES].describe().T
summary["missing"] = march[FEATURES].isna().sum()
summary["missing_pct"] = (
    summary["missing"] / len(march) * 100
)

summary

,count,mean,std,min,25%,50%,75%,max,missing,missing_pct
impressions,331437.0,846.790156,4044.514753,0.0,0.0,2.0,216.0,617124.0,0,0.0
clicks,331437.0,2.479602,19.651282,0.0,0.0,0.0,0.0,5668.0,0,0.0
sessions,331437.0,3.921735,25.381883,0.0,0.0,0.0,1.0,2730.0,0,0.0
engaged_sessions,331437.0,0.089160,0.879657,0.0,0.0,0.0,0.0,224.0,0,0.0
engagement_rate,331437.0,0.007072,0.058838,0.0,0.0,0.0,0.0,1.0,0,0.0


## 3. Future / target leakage audit

The following fields are not allowed as model features because they describe future outcomes, labels, or target information.

In [7]:
future_or_target_fields = [
    "april_sessions",
    "target_decline",
    "label",
    "target",
    "outcome",
    "future_sessions",
    "future_clicks",
    "future_impressions",
    "future_engagement"
]

leakage_found = [
    field
    for field in future_or_target_fields
    if field in FEATURES
]

print("Potential future/target fields:")
print(future_or_target_fields)

print("\nFuture/target fields in approved features:")
print(leakage_found)

assert len(leakage_found) == 0

print("\nFuture/target leakage check PASSED.")

Potential future/target fields:
['april_sessions', 'target_decline', 'label', 'target', 'outcome', 'future_sessions', 'future_clicks', 'future_impressions', 'future_engagement']

Future/target fields in approved features:
[]

Future/target leakage check PASSED.


## 4. Identifier leakage audit

Client/content identifiers are retained only for grouping and traceability. They are not model features.

In [8]:
IDENTIFIERS = [
    "client_hash_id",
    "content_hash_id",
    "report_date"
]

identifier_overlap = [
    field
    for field in IDENTIFIERS
    if field in FEATURES
]

print("Identifier fields:")
print(IDENTIFIERS)

print("\nIdentifiers in approved features:")
print(identifier_overlap)

assert len(identifier_overlap) == 0

print("\nIdentifier leakage check PASSED.")

Identifier fields:
['client_hash_id', 'content_hash_id', 'report_date']

Identifiers in approved features:
[]

Identifier leakage check PASSED.


## 5. Availability / product metadata exclusion

Availability flags and other metadata are excluded from the model feature vector.

In [9]:
excluded_metadata = [
    "client_has_gsc",
    "client_has_ga4",
    "gsc_data_available",
    "ga4_data_available",
    "month"
]

metadata_overlap = [
    field
    for field in excluded_metadata
    if field in FEATURES
]

print("Excluded metadata:")
print(excluded_metadata)

print("\nMetadata fields in approved features:")
print(metadata_overlap)

assert len(metadata_overlap) == 0

print("\nMetadata exclusion check PASSED.")

Excluded metadata:
['client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'month']

Metadata fields in approved features:
[]

Metadata exclusion check PASSED.


## 6. Final feature matrix validation

This is the important validation step. We validate the **cleaned final model input**, not the raw warehouse.

In [10]:
X = march[FEATURES].copy()

for column in FEATURES:
    X[column] = pd.to_numeric(
        X[column],
        errors="coerce"
    )

X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

X = X.fillna(0)

X = X.astype(float)

missing_count = int(
    X.isna().sum().sum()
)

infinite_count = int(
    np.isinf(X.to_numpy(dtype=float)).sum()
)

print("Feature matrix shape:", X.shape)
print("Missing values:", missing_count)
print("Infinite values:", infinite_count)

assert list(X.columns) == FEATURES
assert missing_count == 0
assert infinite_count == 0

print("\nFinal feature matrix validation PASSED.")

Feature matrix shape: (331437, 5)
Missing values: 0
Infinite values: 0

Final feature matrix validation PASSED.


In [11]:
invalid_engagement = X[
    (X["engagement_rate"] < 0) |
    (X["engagement_rate"] > 1)
]

print(
    "Invalid engagement-rate rows:",
    len(invalid_engagement)
)

assert len(invalid_engagement) == 0

print("Engagement-rate validation PASSED.")

Invalid engagement-rate rows: 0
Engagement-rate validation PASSED.


## Final ML-05 decision

**Approved model features:** the five March aggregate features above.

**Not approved as model features:** identifiers, April/future outcome fields, target/label fields, availability/product metadata, and other non-feature fields.

Missing numeric feature values are handled during feature preparation and are not treated as leakage.

This audit does not claim that the source contains no sensitive information. It documents that the selected model feature matrix excludes the identified identifiers and future outcome fields.

In [12]:
audit = {
    "feature_window": "March 2026",
    "approved_features": FEATURES,
    "rows": int(len(X)),
    "missing_values_before_cleaning": int(missing_before.sum()),
    "missing_values_final": missing_count,
    "infinite_values_final": infinite_count,
    "future_target_fields_in_features": leakage_found,
    "identifier_fields_in_features": identifier_overlap,
    "metadata_fields_in_features": metadata_overlap,
    "engagement_rate_valid": True,
    "decision": "APPROVED"
}

output_path = "work/outputs/w03_feature_leakage_audit.json"

with open(output_path, "w") as f:
    json.dump(audit, f, indent=2)

print("Saved:", output_path)
print(json.dumps(audit, indent=2))

Saved: work/outputs/w03_feature_leakage_audit.json
{
  "feature_window": "March 2026",
  "approved_features": [
    "impressions",
    "clicks",
    "sessions",
    "engaged_sessions",
    "engagement_rate"
  ],
  "rows": 331437,
  "missing_values_before_cleaning": 0,
  "missing_values_final": 0,
  "infinite_values_final": 0,
  "future_target_fields_in_features": [],
  "identifier_fields_in_features": [],
  "metadata_fields_in_features": [],
  "engagement_rate_valid": true,
  "decision": "APPROVED"
}


In [13]:
assert len(FEATURES) == 5
assert len(leakage_found) == 0
assert len(identifier_overlap) == 0
assert len(metadata_overlap) == 0
assert X.shape[1] == 5
assert X.isna().sum().sum() == 0
assert np.isfinite(X.to_numpy(dtype=float)).all()
assert (
    (X["engagement_rate"] >= 0) &
    (X["engagement_rate"] <= 1)
).all()

print("========================================")
print("ML-05 W03 FEATURE LEAKAGE AUDIT")
print("========================================")
print("Status: PASSED")
print("Feature window: March 2026")
print("Rows:", len(X))
print("Features:", FEATURES)
print("Missing values:", int(X.isna().sum().sum()))
print("Infinite values:", int(np.isinf(X.to_numpy()).sum()))
print("Future/target leakage:", leakage_found)
print("Identifier leakage:", identifier_overlap)
print("Metadata leakage:", metadata_overlap)
print("Output:", output_path)
print("========================================")

ML-05 W03 FEATURE LEAKAGE AUDIT
Status: PASSED
Feature window: March 2026
Rows: 331437
Features: ['impressions', 'clicks', 'sessions', 'engaged_sessions', 'engagement_rate']
Missing values: 0
Infinite values: 0
Future/target leakage: []
Identifier leakage: []
Metadata leakage: []
Output: work/outputs/w03_feature_leakage_audit.json
